In [ ]:
# =====================================================
# 0. 라이브러리 / 데이터 불러오기
# =====================================================

import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from collections import Counter
from kiwipiepy import Kiwi

INPUT_FILE = "liverpool_cafe_bertopic_macro_result.xlsx"

# 보통 이 시트에 최종 데이터가 있음
topic_df = pd.read_excel(INPUT_FILE, sheet_name="topic_macro_data")

print(topic_df.shape)
print(topic_df.columns.tolist())

# 사용할 텍스트 컬럼
TEXT_COL = "topic_text" if "topic_text" in topic_df.columns else "text_clean"

# 색상
TOTAL_COLOR = "#E8E8E8"
TOPIC_COLOR = "#C8102E"

# 한글 폰트
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# =====================================================
# 1. 토큰화 함수
# =====================================================

kiwi = Kiwi()

# 필요하면 팀별 custom_words 추가
custom_words = [
    # 팀/팬덤
    "리버풀", "리버풀FC", "안필드", "콥", "더콥", "YNWA",

    # 감독/전술
    "슬롯", "아르네슬롯", "클롭", "위르겐클롭",

    # 주요 선수
    "알리송", "알리송베케르",
    "조고메즈", "고메즈",
    "엔도", "엔도와타루",
    "반다이크", "버질반다이크",
    "코나테",
    "케르케즈",
    "비르츠", "플로리안비르츠",
    "소보슬라이", "소보슬러이",
    "이삭", "이사크",
    "맥알리스터", "맥알",
    "살라", "모하메드살라",
    "브래들리",
    "키에사",
    "레오니",
    "커티스존스", "커존",
    "각포", "학포", "가크포",
    "에키티케",
    "마마르다슈빌리", "마마슈",
    "로버트슨",
    "우드먼",
    "프림퐁",
    "그라벤베르흐", "흐라벤베르흐",
    "트레이뇨니",
    "리오은구모하"

    # 조롱/밈 표현
    "리중딱", "살인풀", "전범풀", "뎀바바", "소라드",
    "훔바", "훔바훔바", "리버풀조롱"
]

for word in custom_words:
    kiwi.add_user_word(word, "NNP")

stopwords = set([
    # [1] 기존 & 유튜브/커뮤니티 은어
    '진짜', '너무', '영상', '오늘', '경기', '그냥', '진심', 'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', 'ㅜㅜ', '댓글', '개인',
    '이거', '저거', '그거', '뭔가', '정말', '지금', '다시', '계속', '가장', '제일', '조금', '정도','개추',
    
    # [2] 축구 일반 용어 (분석 인사이트가 없는 뻔한 단어)
    '축구', '하이라이트', '감독', '선수', '시즌', '리그', '전반', '후반', '골', '볼', '팬', '응원', '생각', '사람', '우리', '이번',
    
    # [3] 리버풀 지칭
    '리버풀', '리중딱', '전범풀',
    
    # [4] 타 팀 및 관련 선수 (노이즈 제거)
    '맨유', '맨체스터유나이티드', '맹구', '맨시티', '맨체스터시티', 
    '아스날', '아스널', '개스널', '리버풀', '전범풀', '첼시', '첼강딱', 
    '뉴캐슬', '빌라', '에버튼', '울버햄튼', '황희찬', 
    '레알', '바르샤', '뮌헨', '김민재', '이강인', '파리', 'psg', '리즈',
    '웨스트햄', '아스톤빌라','웨햄','클럽',
    
    
    # [6] 형태소 분석(Okt) 후 흔히 남는 무의미한 두 글자 명사들
    '때문', '위해', '대한', '관련', '진짜' ,"작성","짧은","글", "질문", "삭제", "필수", "검색", "경기시작", "익일", "오전" ,"일반","일반글", "작성", "금지","게시","필독"
])

def kiwi_tokenizer(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in kiwi.tokenize(text):
        if token.tag in ["NNG", "NNP", "SL"]:
            word = token.form.strip().lower()

            if len(word) >= 2 and word not in stopwords:
                tokens.append(word)

    return tokens


def count_keywords(data):
    counter = Counter()

    for text in data[TEXT_COL].dropna().astype(str):
        counter.update(kiwi_tokenizer(text))

    return counter

In [ ]:
# =====================================================
# 2. 상위 3개 macro_topic 키워드 빈도 비교
#    - 전체 데이터 빈도 vs 해당 토픽 내부 빈도
# =====================================================

def plot_macro_keyword_bar(macro_topic, top_n=12):
    total_counter = count_keywords(topic_df)

    target_df = topic_df[topic_df["macro_topic"] == macro_topic].copy()
    topic_counter = count_keywords(target_df)

    top_words = [w for w, _ in topic_counter.most_common(top_n)]

    plot_df = pd.DataFrame({
        "keyword": top_words,
        "total_count": [total_counter[w] for w in top_words],
        "topic_count": [topic_counter[w] for w in top_words]
    })

    plot_df = plot_df.sort_values("topic_count", ascending=True)

    plt.figure(figsize=(9, 5.5))

    plt.barh(
        plot_df["keyword"],
        plot_df["total_count"],
        color=TOTAL_COLOR,
        label="전체 데이터 빈도"
    )

    plt.barh(
        plot_df["keyword"],
        plot_df["topic_count"],
        color=TOPIC_COLOR,
        label=f"{macro_topic} 내부 빈도"
    )

    plt.title(f"{macro_topic} 주요 키워드 빈도", fontsize=15, fontweight="bold")
    plt.xlabel("Keyword Frequency")
    plt.ylabel("")
    plt.grid(axis="x", linestyle="--", alpha=0.25)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


# 기타/분류불가 제외하고 상위 3개 macro_topic 선택
exclude_topics = ["기타", "기타/분류불가"]

top3_topics = (
    topic_df[~topic_df["macro_topic"].isin(exclude_topics)]
    ["macro_topic"]
    .value_counts()
    .head(3)
    .index
    .tolist()
)

for topic in top3_topics:
    plot_macro_keyword_bar(topic, top_n=12)

In [ ]:
# # =====================================================
# # 3. 세로 막대그래프 + 점/라인 + 비율 라벨 함수
# # =====================================================

# BAR_COLOR = "#132257"
# LINE_COLOR = "#E6B325"

# def plot_single_topic_ratio(data, title, top_n=8, save_path=None):
#     ratio = (
#         data["macro_topic"]
#         .value_counts(normalize=True)
#         .mul(100)
#         .head(top_n)
#         .sort_values(ascending=False)
#     )

#     plot_df = ratio.reset_index()
#     plot_df.columns = ["macro_topic", "ratio"]

#     x = range(len(plot_df))

#     plt.figure(figsize=(10, 5.5))

#     plt.bar(
#         x,
#         plot_df["ratio"],
#         color=BAR_COLOR,
#         alpha=0.88,
#         width=0.62
#     )

#     plt.plot(
#         x,
#         plot_df["ratio"],
#         color=LINE_COLOR,
#         marker="o",
#         linewidth=2.4,
#         markersize=7
#     )

#     for i, v in enumerate(plot_df["ratio"]):
#         plt.text(
#             i,
#             v + 0.8,
#             f"{v:.1f}%",
#             ha="center",
#             va="bottom",
#             fontsize=10,
#             fontweight="bold"
#         )

#     plt.title(title, fontsize=16, fontweight="bold", pad=15)
#     plt.ylabel("Ratio (%)", fontsize=11)
#     plt.xticks(x, plot_df["macro_topic"], rotation=35, ha="right")
#     plt.ylim(0, max(plot_df["ratio"]) * 1.22)

#     plt.grid(axis="y", linestyle="--", alpha=0.25)
#     plt.gca().spines["top"].set_visible(False)
#     plt.gca().spines["right"].set_visible(False)
#     plt.gca().spines["left"].set_alpha(0.3)
#     plt.gca().spines["bottom"].set_alpha(0.3)

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=300, bbox_inches="tight")

#     plt.show()

In [ ]:
# # =====================================================
# # 4. 38R Before / After Topic 비율
# # =====================================================

# before_df = topic_df[topic_df["period"] == "before_38R"].copy()
# after_df = topic_df[topic_df["period"] == "after_38R"].copy()

# plot_single_topic_ratio(
#     before_df,
#     title="Before 38R Topic Ratio",
#     top_n=8,
#     save_path="tottenham_before_38R_topic_ratio.png"
# )

# plot_single_topic_ratio(
#     after_df,
#     title="After 38R Topic Ratio",
#     top_n=8,
#     save_path="tottenham_after_38R_topic_ratio.png"
# )

In [ ]:
# # =====================================================
# # 5. Positive / Negative Topic 비율
# # =====================================================

# positive_df = topic_df[topic_df["gpt_sentiment"] == "positive"].copy()
# negative_df = topic_df[topic_df["gpt_sentiment"] == "negative"].copy()

# plot_single_topic_ratio(
#     positive_df,
#     title="Positive Topic Ratio",
#     top_n=8,
#     save_path="tottenham_positive_topic_ratio.png"
# )

# plot_single_topic_ratio(
#     negative_df,
#     title="Negative Topic Ratio",
#     top_n=8,
#     save_path="tottenham_negative_topic_ratio.png"
# )

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 색상
MAIN_COLOR = "#C8102E"
SUB_COLOR = "#E8E8E8"
LINE_COLORS = [
    "#C8102E",
    "#E31C3D",
    "#A50021",
    "#D4AF37",
    "#8B0000",
    "#6B7280",
    "#F4A6B7",
    "#4A4A4A"
]

# 한글 폰트
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


def make_before_after_ratio(topic_df, top_n=6, exclude_topics=None):
    data = topic_df[
        topic_df["period"].isin(["before_38R", "after_38R"])
    ].copy()

    if exclude_topics:
        data = data[~data["macro_topic"].isin(exclude_topics)]

    # 전체 기준 상위 토픽 순서 고정
    top_topics = data["macro_topic"].value_counts().head(top_n).index.tolist()

    data["macro_topic_plot"] = data["macro_topic"].where(
        data["macro_topic"].isin(top_topics),
        "기타"
    )

    # 기타 포함 후 순서 고정
    topic_order = top_topics + (["기타"] if "기타" in data["macro_topic_plot"].values else [])

    ratio_df = (
        pd.crosstab(
            data["macro_topic_plot"],
            data["period"],
            normalize="columns"
        )
        .mul(100)
        .reindex(topic_order)
        .fillna(0)
    )

    ratio_df = ratio_df.rename(columns={
        "before_38R": "Before 38R",
        "after_38R": "After 38R"
    })

    return ratio_df

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

def plot_before_after_pie_with_center_legend(topic_df, top_n=6, exclude_topics=None, save_path=None):
    ratio_df = make_before_after_ratio(
        topic_df,
        top_n=top_n,
        exclude_topics=exclude_topics
    )

    labels = ratio_df.index.tolist()
    colors = LINE_COLORS[:len(labels)]

    fig = plt.figure(figsize=(14, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.6, 1])

    ax1 = fig.add_subplot(gs[0, 0])
    ax_leg = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[0, 2])

    # Before
    wedges1, texts1, autotexts1 = ax1.pie(
        ratio_df["Before 38R"].values,
        labels=None,
        autopct=lambda p: f"{p:.1f}%" if p >= 4 else "",
        startangle=90,
        counterclock=False,
        colors=colors,
        pctdistance=0.72,
        wedgeprops={"edgecolor": "white", "linewidth": 1.2}
    )
    ax1.set_title("Before 38R", fontsize=15, fontweight="bold")

    # After
    wedges2, texts2, autotexts2 = ax2.pie(
        ratio_df["After 38R"].values,
        labels=None,
        autopct=lambda p: f"{p:.1f}%" if p >= 4 else "",
        startangle=90,
        counterclock=False,
        colors=colors,
        pctdistance=0.72,
        wedgeprops={"edgecolor": "white", "linewidth": 1.2}
    )
    ax2.set_title("After 38R", fontsize=15, fontweight="bold")

    # 퍼센트 글씨
    for autotext in autotexts1 + autotexts2:
        autotext.set_fontsize(9)
        autotext.set_fontweight("bold")
        autotext.set_color("white")

    # 가운데 축에 범례만 넣기
    ax_leg.axis("off")
    ax_leg.legend(
        wedges1,
        labels,
        title="Macro Topic",
        loc="center",
        frameon=False,
        fontsize=9,
        title_fontsize=10
    )

    fig.suptitle("Topic Composition Before and After 38R", fontsize=17, fontweight="bold")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

plot_before_after_pie_with_center_legend(
    topic_df,
    top_n=6,
    exclude_topics=["기타/분류불가", "기타"],
    save_path="tottenham_before_after_pie_center_legend.png"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================
# Before / After Topic Ratio Line Plot
# x축: topic 고정 / line: before, after
# =====================================================

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

BEFORE_COLOR = "#E8E8E8"
AFTER_COLOR = "#C8102E"

def plot_before_after_topic_line(topic_df, top_n=8, exclude_topics=None, save_path=None):
    data = topic_df[
        topic_df["period"].isin(["before_38R", "after_38R"])
    ].copy()

    if exclude_topics:
        data = data[~data["macro_topic"].isin(exclude_topics)]

    # 전체 기준 상위 topic 순서 고정
    topic_order = (
        data["macro_topic"]
        .value_counts()
        .head(top_n)
        .index
        .tolist()
    )

    ratio_df = (
        pd.crosstab(
            data["macro_topic"],
            data["period"],
            normalize="columns"
        )
        .mul(100)
        .reindex(topic_order)
        .fillna(0)
    )

    ratio_df = ratio_df.rename(columns={
        "before_38R": "Before 38R",
        "after_38R": "After 38R"
    })

    x = range(len(ratio_df))

    plt.figure(figsize=(11, 5.8))

    plt.plot(
        x,
        ratio_df["Before 38R"],
        marker="o",
        linewidth=2.8,
        markersize=7,
        color=BEFORE_COLOR,
        label="Before 38R"
    )

    plt.plot(
        x,
        ratio_df["After 38R"],
        marker="o",
        linewidth=2.8,
        markersize=7,
        color=AFTER_COLOR,
        label="After 38R"
    )

    # 비율 라벨
    for i, topic in enumerate(ratio_df.index):
        before_v = ratio_df.loc[topic, "Before 38R"]
        after_v = ratio_df.loc[topic, "After 38R"]

        plt.text(
            i,
            before_v + 0.7,
            f"{before_v:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            color=BEFORE_COLOR,
            fontweight="bold"
        )

        plt.text(
            i,
            after_v + 0.7,
            f"{after_v:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            color=AFTER_COLOR,
            fontweight="bold"
        )

    plt.xticks(
        x,
        ratio_df.index,
        rotation=35,
        ha="right",
        fontsize=10
    )

    plt.title(
        "Topic Ratio Comparison: Before vs After 38R",
        fontsize=16,
        fontweight="bold",
        pad=15
    )

    plt.ylabel("Topic Ratio (%)", fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.25)
    plt.legend(frameon=False, fontsize=10)

    plt.gca().spines["top"].set_visible(False)
    plt.gca().spines["right"].set_visible(False)
    plt.gca().spines["left"].set_alpha(0.3)
    plt.gca().spines["bottom"].set_alpha(0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return ratio_df

In [ ]:
ratio_df = plot_before_after_topic_line(
    topic_df,
    top_n=8,
    exclude_topics=["기타/분류불가", "기타"],
    save_path="tottenham_before_after_topic_line_by_topic.png"
)

display(ratio_df.round(2))